In [ ]:
pip install torch torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool

In [ ]:
def add_3d_features(data):
    src, tgt = data.edge_index          # row=source atom, col=target atom


    diff = data.pos[src] - data.pos[tgt]          # [E, 3]  raw vector A→B
    dist = torch.norm(diff, dim=-1, keepdim=True) # [E, 1]
    unit_vec = diff / (dist + 1e-8)               # [E, 3]  avoid divide-by-zero
    num_rbf  = 16
    centers  = torch.linspace(0.0, 5.0, num_rbf) # [16]
    width    = (centers[1] - centers[0]).item()   # spacing between centers
    rbf      = torch.exp(
                   -((dist - centers) ** 2) / (width ** 2)
               )                                  # [E, 16]
    data.edge_attr = torch.cat([
        data.edge_attr,   # [E, 4]  original bond features
        unit_vec,         # [E, 3]  direction
        rbf               # [E, 16] distance encoded as gaussians
    ], dim=-1)

    return data

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import importlib
import torch_geometric.datasets.qm9 as qm9_module
import torch
torch.manual_seed(42)
importlib.reload(qm9_module)
from torch_geometric.datasets import QM9
dataset = QM9(root='data/QM9',transform=add_3d_features).shuffle()

Extracting data/QM9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


In [ ]:
train_data = dataset[:104000]
test_data  = dataset[104000:]

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=32)

In [ ]:
train_y = torch.cat([data.y[:, :16] for data in train_data], dim=0)
target_mean = train_y.mean(dim=0).to(device)
target_std  = train_y.std(dim=0).to(device)
target_std[target_std == 0] = 1.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool

In [ ]:
class UGNNMolecule(nn.Module):
    def __init__(
        self,
        node_dim  = 11,    # raw input features per atom
        edge_dim  = 23,    # bond features (4 original + 3 direction + 16 RBF)
        hidden    = 64,    # base hidden size — encoder will grow from this
        out_dim   = 16,    # number of molecular properties to predict
    ):
        super().__init__()
        self.node_encoder = nn.Sequential(
            nn.Linear(node_dim, hidden),   # the actual projection matrix
            nn.ReLU(),                     # non-linearity
            nn.Linear(hidden, hidden),     # one extra layer for richer encoding
            nn.ReLU()
        )
        self.enc1=GINEConv(nn=nn.Sequential(
            nn.Linear(hidden,hidden*2),
            nn.ReLU(),
            nn.Linear(hidden*2,hidden*2)
        ),edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden * 2)
        self.enc2=GINEConv(nn=nn.Sequential(
            nn.Linear(hidden*2,hidden*4),
            nn.ReLU(),
            nn.Linear(hidden*4,hidden*4)
        ),edge_dim=edge_dim)

        self.bn2 = nn.BatchNorm1d(hidden * 4)
        self.bottleneck = GINEConv(
            nn = nn.Sequential(
                nn.Linear(hidden*4, hidden*4),   # ← pointwise: transform features
                nn.ReLU(),
                nn.Linear(hidden*4, hidden*4)    # ← pointwise: transform features back
            ),
            edge_dim = edge_dim        # ← depthwise equivalent: message passing over bonds
        )
        self.bn_bottleneck = nn.BatchNorm1d(hidden * 4)
        self.dec2 = GINEConv(
            nn = nn.Sequential(
                nn.Linear(hidden * 4 + hidden * 2, hidden * 2),  # 384 → 128
                nn.ReLU(),
                nn.Linear(hidden * 2, hidden * 2)                # 128 → 128
            ),
            edge_dim = edge_dim
        )
        self.bn_dec2 = nn.BatchNorm1d(hidden * 2)
        self.dec1=GINEConv(
            nn=nn.Sequential(
                nn.Linear(hidden*2+hidden,hidden),
                nn.ReLU(),
                nn.Linear(hidden,hidden)
            ),edge_dim=edge_dim
        )
        self.bn_dec1=nn.BatchNorm1d(hidden)
        self.output=nn.Sequential(
            nn.Linear(hidden,hidden),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden,out_dim)

        )


    def forward(self, data):
        x          = data.x           # [num_atoms, 11]  raw node features
        edge_index = data.edge_index  # [2, num_edges]   connectivity
        edge_attr  = data.edge_attr   # [num_edges, 23]  bond features
        batch      = data.batch       # [num_atoms]      which graph each atom belongs to

        # ── Step 0: project node features into hidden space ───────────────────
        x = self.node_encoder(x)      # [num_atoms, 11] → [num_atoms, 64]
        skip1=x
        x=self.enc1(x,edge_index,edge_attr)
        x = self.bn1(x)
        x = F.relu(x)
        skip2=x
        x=self.enc2(x,edge_index,edge_attr)
        x=self.bn2(x)
        x=F.relu(x)
        x = self.bottleneck(x, edge_index, edge_attr)
        x = self.bn_bottleneck(x)
        x = F.relu(x)
        x=torch.cat([x,skip2],dim=1)
        x = self.dec2(x, edge_index, edge_attr)
        x = self.bn_dec2(x)
        x = F.relu(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.dec1(x, edge_index, edge_attr)
        x = self.bn_dec1(x)
        x = F.relu(x)
        x=global_mean_pool(x,batch)
        x=self.output(x)



        return x   # [num_atoms, 64] — temporary, just to verify shape




In [ ]:
model = UGNNMolecule().to(device)
# .to(device) moves all weights to GPU if available

optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)
# Adam adjusts each weight's learning rate individually
# lr=4e-4 means weights move 0.0004 * gradient each step

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = 'min',    # we want test MAE to go DOWN
    factor   = 0.5,      # halve the learning rate when plateauing
    patience = 5,        # wait 5 epochs of no improvement before reducing
    min_lr   = 1e-5      # never go below this
)

In [ ]:
def train(loader):
    model.train()

    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        true = (batch.y[:, :16] - target_mean) / target_std
        loss = F.mse_loss(pred, true)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)
    # average loss per molecule across the whole epoch

In [ ]:
@torch.no_grad()
def test(loader):
    model.eval()

    total_mae = 0

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch) * target_std + target_mean

        true = batch.y[:, :16]

        total_mae += F.l1_loss(pred, true, reduction='sum').item()

    return total_mae / len(loader.dataset)
    # average MAE per molecule

In [ ]:
print(f"{'Epoch':>5}  {'Train MSE':>10}  {'Test MAE':>10}  {'LR':>10}")
print("-" * 45)

for epoch in range(1, 151):

    train_loss = train(train_loader)
    # runs all batches, updates weights, returns average loss

    test_mae = test(test_loader)
    # runs all test batches, no weight updates, returns average MAE

    scheduler.step(test_mae)
    # hand test MAE to scheduler
    # it decides whether to reduce learning rate

    if epoch % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"{epoch:>5}  {train_loss:>10.4f}  {test_mae:>10.4f}  {current_lr:>10.6f}")

Epoch   Train MSE    Test MAE          LR
---------------------------------------------
    5      0.1067    972.4212    0.000400
   10      0.0761    593.2737    0.000400
   15      0.0651    665.1306    0.000400
   20      0.0530    282.2382    0.000200
   25      0.0505    357.8407    0.000200
   30      0.0486    298.9146    0.000200
   35      0.0472    273.6809    0.000200
   40      0.0460    631.3718    0.000200
   45      0.0418    222.1608    0.000100
   50      0.0413    209.7193    0.000100
   55      0.0405    229.1082    0.000100
   60      0.0399    214.5518    0.000100
   65      0.0386    196.5818    0.000050
   70      0.0380    193.9143    0.000050
   75      0.0372    205.4238    0.000025
   80      0.0366    182.4392    0.000025
   85      0.0366    183.6674    0.000013
   90      0.0361    180.6678    0.000010
   95      0.0362    185.5568    0.000010
  100      0.0360    176.1841    0.000010
  105      0.0361    174.4951    0.000010
  110      0.0361    177.1204 

In [ ]:
print("hi")

hi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

torch.save({
    'model_state_dict': model.state_dict(),
    'target_mean':      target_mean,
    'target_std':       target_std,
}, '/content/drive/MyDrive/ugnn_final.pth')

print("✅ Final model saved")

Mounted at /content/drive
✅ Final model saved


In [ ]:
from google.colab import files
uploaded = files.upload()   # a file picker will appear — select your .pth file

Saving ugnn_final (1).pth to ugnn_final (1).pth


In [ ]:
# ── Step 1: load the saved model ─────────────────────────────────────────────
checkpoint = torch.load('ugnn_final.pth',map_location=device)

model = UGNNMolecule().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

target_mean = checkpoint['target_mean'].to(device)
target_std  = checkpoint['target_std'].to(device)

# ── Step 2: pick any molecule from the test set ───────────────────────────────
sample = test_data[0]          # grab one molecule
sample = sample.to(device)

# ── Step 3: predict ───────────────────────────────────────────────────────────
with torch.no_grad():
    pred = model(sample)
    pred = pred * target_std + target_mean    # denormalise back to real units

# ── Step 4: compare side by side ──────────────────────────────────────────────
property_names = [
    'Dipole moment',
    'Isotropic polarizability',
    'HOMO energy',
    'LUMO energy',
    'HOMO-LUMO gap',
    'Electronic spatial extent',
    'ZPVE',
    'U0',
    'U',
    'H',
    'G',
    'Cv',
    'U0_atom',
    'U_atom',
    'H_atom',
    'G_atom'
]

true = sample.y[:, :16].squeeze()     # ground truth for this molecule
pred = pred.squeeze()                 # model prediction

print(f"{'Property':<30}  {'Predicted':>12}  {'True':>12}  {'Error':>12}")
print("-" * 70)
for i, name in enumerate(property_names):
    predicted = pred[i].item()
    actual    = true[i].item()
    error     = abs(predicted - actual)
    print(f"{name:<30}  {predicted:>12.4f}  {actual:>12.4f}  {error:>12.4f}")

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()

    all_preds = []
    all_true  = []

    for batch in loader:
        batch = batch.to(device)
        pred  = model(batch) * target_std + target_mean   # denormalise
        true  = batch.y[:, :16]

        all_preds.append(pred.cpu())
        all_true.append(true.cpu())

    all_preds = torch.cat(all_preds, dim=0)   # [N, 16]
    all_true  = torch.cat(all_true,  dim=0)   # [N, 16]

    # MAE per property
    mae  = (all_preds - all_true).abs().mean(dim=0)

    # RMSE per property
    rmse = ((all_preds - all_true) ** 2).mean(dim=0).sqrt()

    # R² per property
    ss_res = ((all_true - all_preds) ** 2).sum(dim=0)
    ss_tot = ((all_true - all_true.mean(dim=0)) ** 2).sum(dim=0)
    r2     = 1 - ss_res / ss_tot

    property_names = [
        'Dipole moment',            'Isotropic polarizability',
        'HOMO energy',              'LUMO energy',
        'HOMO-LUMO gap',            'Electronic spatial extent',
        'ZPVE',                     'U0',
        'U',                        'H',
        'G',                        'Cv',
        'U0_atom',                  'U_atom',
        'H_atom',                   'G_atom'
    ]

    print(f"{'Property':<30}  {'MAE':>10}  {'RMSE':>10}  {'R²':>10}")
    print("-" * 65)
    for i, name in enumerate(property_names):
        print(f"{name:<30}  {mae[i]:>10.4f}  {rmse[i]:>10.4f}  {r2[i]:>10.4f}")

    return mae, rmse, r2


# run it
mae, rmse, r2 = evaluate(test_loader)

Property                               MAE        RMSE          R²
-----------------------------------------------------------------
Dipole moment                       0.4187      0.5801      0.8488
Isotropic polarizability            0.5652      0.7788      0.9908
HOMO energy                         0.0989      0.1285      0.9536
LUMO energy                         0.1205      0.1562      0.9851
HOMO-LUMO gap                       0.1484      0.1942      0.9773
Electronic spatial extent          23.0962     30.2349      0.9883
ZPVE                                0.0465      0.0601      0.9956
U0                                 36.6873     51.7549      0.9977
U                                  36.6871     51.7546      0.9977
H                                  36.6871     51.7546      0.9977
G                                  36.6875     51.7552      0.9977
Cv                                  0.2513      0.3246      0.9937
U0_atom                             0.3464      0.4653      0.9